In [ ]:
import numpy as np
import scipy
import utils


In [ ]:
def get_atm_center(ds, sfc_center):
    
    nan_mask = ds['prmsl'].isnull()
    ds_sfc = ds.where(~nan_mask, drop = True).isel(latitude = slice(50,-50), longitude = slice(50,-50))
    

    slp_threshold = np.quantile(ds_sfc['prmsl'], 0.01)
    slp_clusters = ds_sfc['prmsl'].where(ds_sfc['prmsl'] < slp_threshold)
    slp_clusters_mask = ~slp_clusters.isnull()

    clusters = scipy.ndimage.label(slp_clusters_mask)
    cluster_size = []
    for a in np.arange(clusters[1]):
        cluster_number = a + 1
        cluster_size.append(np.sum(np.where(clusters[0] == cluster_number, 1, 0)))

    
    largest_cluster = np.argmax(cluster_size) + 1
    center_y, center_x = scipy.ndimage.center_of_mass(clusters[0], largest_cluster)
    mslp = ds_sfc['prmsl'].where(clusters[0] == largest_cluster).min().values

    center_lat = ds_sfc.latitude.isel(latitude =int(np.round(center_y)))
    center_lon = ds_sfc.longitude.isel(longitude =int(np.round(center_x)))
    
    center_info = dict(center_lat = center_lat, center_lon = center_lon, mslp = mslp)
    return center_info

def get_sfc_center(ds):
    min_pressure = ds['prmsl'].min().values
    if min_pressure > 1005: # Pressure threshold (pa) 1005mb
        return None
    nan_mask = ds['prmsl'].isnull()
    ds_sfc = ds.where(~nan_mask, drop = True).isel(latitude = slice(50,-50), longitude = slice(50,-50))
    

    slp_threshold = np.quantile(ds_sfc['prmsl'], 0.01)
    slp_clusters = ds_sfc['prmsl'].where(ds_sfc['prmsl'] < slp_threshold)
    slp_clusters_mask = ~slp_clusters.isnull()

    clusters = scipy.ndimage.label(slp_clusters_mask)
    cluster_size = []
    for a in np.arange(clusters[1]):
        cluster_number = a + 1
        cluster_size.append(np.sum(np.where(clusters[0] == cluster_number, 1, 0)))

    
    largest_cluster = np.argmax(cluster_size) + 1
    center_y, center_x = scipy.ndimage.center_of_mass(clusters[0], largest_cluster)
    mslp = ds_sfc['prmsl'].where(clusters[0] == largest_cluster).min().values

    center_lat = ds_sfc.latitude.isel(latitude =int(np.round(center_y)))
    center_lon = ds_sfc.longitude.isel(longitude =int(np.round(center_x)))
    
    center_info = dict(center_lat = center_lat, center_lon = center_lon, mslp = mslp)
    return center_info

In [ ]:
url = 'https://noaa-nws-hafs-pds.s3.amazonaws.com/hfsa/20230927/12/17l.2023092712.hfsa.storm.atm.f024.grb2'

atm_args = dict(typeOfKey = 'isobaricInhPa',filename="Model_Data/temp_gribs/atm_temp.grib2" )
sfc_args = dict(typeOfKey = 'meanSea',filename="Model_Data/temp_gribs/sfc_temp.grb2" )

In [ ]:
sfc_ds = utils.download_and_open(url, **sfc_args)